# Drishti VTOE Worker - Google Colab T4

Runs the in-house Virtual Try-On Engine on NVIDIA T4 (16GB VRAM).

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy the public URL from the last cell
4. Set up Cloudflare Tunnel for external access

In [ ]:
#@title 1. Check GPU
!nvidia-smi

In [ ]:
#@title 2. Install dependencies (run once)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers peft accelerate xformers
!pip install -q insightface onnxruntime-gpu
!pip install -q controlnet-aux ultralytics
!pip install -q fastapi uvicorn pydantic python-multipart
!pip install -q opencv-python-headless structlog
!pip install -q basicsr facexlib
!pip install -q clean-fid scikit-image
!pip install -q pyngrok nest-asyncio

print('Dependencies installed')

In [ ]:
#@title 3. Clone DRISHTI repo
!git clone https://github.com/Shuttler14/mynarrative-drishti.git
%cd mynarrative-drishti

# Download CodeFormer weights
!mkdir -p /content/models/codeformer
!wget -q -O /content/models/codeformer/codeformer.pth \
  https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth

print('Repo cloned, CodeFormer weights downloaded')

In [ ]:
#@title 4. Start VTOE server + Cloudflare Tunnel
import threading, time, subprocess, sys

# Start FastAPI server
def run_server():
    !uvicorn vtoe.api.server:app --host 0.0.0.0 --port 8001 --log-level info

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Start Cloudflare Tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8001'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)

# Extract tunnel URL
import re
tunnel_url = None
for line in iter(tunnel.stderr.readline, ''):
    if 'trycloudflare.com' in str(line):
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', str(line))
        if match:
            tunnel_url = match.group(0)
            break

print(f'VTOE Server running on: {tunnel_url}')
print(f'Health check: {tunnel_url}/v1/health')
print(f'Try-on endpoint: {tunnel_url}/v1/try-on')

# Store for heartbeat
VTOE_URL = tunnel_url

In [ ]:
#@title 5. Heartbeat (anti-idle + Oracle Cloud registration)
import time, requests, threading

ORACLE_HOST = ''  #@param {type:"string"}

def heartbeat():
    while True:
        try:
            if ORACLE_HOST:
                requests.post(
                    f'https://{ORACLE_HOST}/vtoe/heartbeat',
                    json={'url': VTOE_URL},
                    timeout=5
                )
            # Also check health
            r = requests.get(f'{VTOE_URL}/v1/health', timeout=5)
            print(f'Heartbeat OK - GPU: {r.json().get("gpu_used_gb", 0):.1f}GB used')
        except Exception as e:
            print(f'Heartbeat failed: {e}')
        time.sleep(300)  # 5 min

threading.Thread(target=heartbeat, daemon=True).start()
print('Heartbeat started (every 5 minutes)')

In [ ]:
#@title 6. Test try-on
import requests, base64, io
from PIL import Image

# Create test images
person = Image.new('RGB', (768, 1024), (200, 180, 160))
garment = Image.new('RGB', (768, 1024), (180, 40, 40))

def img_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

resp = requests.post(f'{VTOE_URL}/v1/try-on', json={
    'person_image': img_to_b64(person),
    'garment_image': img_to_b64(garment),
    'garment_type': 'ethnic',
    'ethnic_sub_type': 'saree',
    'quality': 'fast',
    'preserve_face': True
})

if resp.status_code == 200:
    result = resp.json()
    print(f'Status: {result["status"]}')
    print(f'Time: {result["processing_time_ms"]}ms')
    print(f'Quality: {result["quality_score"]}')
    print(f'Face sim: {result["face_similarity"]}')
    print(f'Garment sim: {result["garment_similarity"]}')
else:
    print(f'Error: {resp.status_code} - {resp.text}')

In [ ]:
#@title 7. Keep alive (run this cell to prevent Colab from disconnecting)
import time
while True:
    time.sleep(60)
    print('Still alive...')
    # Keep alive by running a small GPU operation
    import torch
    _ = torch.zeros(1).cuda()
    time.sleep(59)